# 🛍️ Shopping Mall Customer Segmentation
## Comparison — All Methods
### K-Means vs MeanShift vs DBSCAN vs Autoencoder

---
### 📌 What does this notebook do?
After all members have run their individual notebooks, we bring together
all results and compare them using:
1. **Silhouette Score** — how well-separated the clusters are
2. **Davies-Bouldin Index** — how compact and separate clusters are
3. **V-Measure Score** — how consistent the clustering is between methods
4. **Side-by-side 3D plots** — visual comparison
5. **Best algorithm selection** → used for Streamlit UI

> ⚠️ Run ALL member notebooks first (01, 02, 03, 04)!

---
## Step 1: Load All Results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MeanShift, DBSCAN, estimate_bandwidth
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    v_measure_score, homogeneity_score, completeness_score
)
import pickle
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

X_scaled = np.load('data/X_scaled.npy')
df_clean = pd.read_csv('data/data_preprocessed.csv')
df       = pd.read_csv('data/Shopping_Mall_Customer_Segmentation_Data_.csv')

# Load scores from each member's notebook
km_scores  = pickle.load(open('kmeans_scores.pkl',    'rb'))
ms_scores  = pickle.load(open('meanshift_scores.pkl', 'rb'))
db_scores  = pickle.load(open('dbscan_scores.pkl',    'rb'))
ae_scores  = pickle.load(open('autoencoder_scores.pkl', 'rb'))

# Load cluster labels
km_labels  = pickle.load(open('kmeans_labels.pkl',    'rb'))
ms_labels  = pickle.load(open('meanshift_labels.pkl', 'rb'))
db_labels  = pickle.load(open('dbscan_labels.pkl',    'rb'))
ae_labels  = pickle.load(open('autoencoder_labels.pkl', 'rb'))

print('✅ All results loaded!')
print(f'   K-Means    : {km_scores["n_clusters"]} clusters')
print(f'   MeanShift  : {ms_scores["n_clusters"]} clusters')
print(f'   DBSCAN     : {db_scores["n_clusters"]} clusters + {db_scores["noise"]} noise points')
print(f'   Autoencoder: {ae_scores["n_clusters"]} clusters')

---
## Step 2: Metrics Comparison Table

### 📌 How to read this table:
| Metric | What it measures | Winner |
|--------|-----------------|--------|
| Silhouette Score | How well clusters are separated (0–1) | **Highest** |
| Davies-Bouldin | Cluster compactness vs separation | **Lowest** |
| Noise Points | Customers not assigned to any cluster | Lower |
| No. Clusters | Number of groups found | Depends on context |

In [ ]:
all_scores = [km_scores, ms_scores, db_scores, ae_scores]
comparison = pd.DataFrame(all_scores)
comparison.columns = ['Algorithm','No. Clusters','Silhouette Score',
                      'Davies-Bouldin','Inertia','Noise Points'] + \
                     ([col for col in comparison.columns if col not in
                       ['algorithm','n_clusters','silhouette','davies_bouldin','inertia','noise']])

# Clean display
display_df = pd.DataFrame({
    'Algorithm'        : [s['algorithm']      for s in all_scores],
    'No. Clusters'     : [s['n_clusters']     for s in all_scores],
    'Silhouette ↑'     : [s['silhouette']     for s in all_scores],
    'Davies-Bouldin ↓' : [s['davies_bouldin'] for s in all_scores],
    'Noise Points'     : [s['noise']          for s in all_scores],
})

print('=' * 75)
print('                    FINAL COMPARISON TABLE')
print('=' * 75)
print(display_df.to_string(index=False))
print('=' * 75)
print('  ↑ = higher is better    ↓ = lower is better')

---
## Step 3: V-Measure Score — Consistency Between Methods

### 📌 What is V-Measure?
V-Measure checks how **consistent** two clustering methods are with each other.
- Score = 1.0 → the two methods agree perfectly on how to group customers
- Score = 0.0 → the two methods completely disagree

We use K-Means as the reference (since it's the most common baseline),
and compare each other method against it.

**Why this matters:** If two different algorithms find similar clusters,
it means the groupings are REAL natural patterns in the data, not just algorithm artefacts.

In [ ]:
# Use KMeans as reference labels
# For DBSCAN: exclude noise points from comparison
mask_db = db_labels != -1

print('=== V-Measure Scores (vs K-Means as reference) ===')
print()

# KMeans vs MeanShift
vm_ms = v_measure_score(km_labels, ms_labels)
h_ms  = homogeneity_score(km_labels, ms_labels)
c_ms  = completeness_score(km_labels, ms_labels)
print(f'K-Means vs MeanShift:')
print(f'  V-Measure Score : {vm_ms:.4f}  (1.0 = perfect agreement)')
print(f'  Homogeneity     : {h_ms:.4f}  (same customers in same groups?)')
print(f'  Completeness    : {c_ms:.4f}  (all members of a group together?)')
print()

# KMeans vs DBSCAN (non-noise only)
vm_db = v_measure_score(km_labels[mask_db], db_labels[mask_db])
h_db  = homogeneity_score(km_labels[mask_db], db_labels[mask_db])
c_db  = completeness_score(km_labels[mask_db], db_labels[mask_db])
print(f'K-Means vs DBSCAN (non-noise only):')
print(f'  V-Measure Score : {vm_db:.4f}')
print(f'  Homogeneity     : {h_db:.4f}')
print(f'  Completeness    : {c_db:.4f}')
print()

# KMeans vs Autoencoder
vm_ae = v_measure_score(km_labels, ae_labels)
h_ae  = homogeneity_score(km_labels, ae_labels)
c_ae  = completeness_score(km_labels, ae_labels)
print(f'K-Means vs Autoencoder:')
print(f'  V-Measure Score : {vm_ae:.4f}')
print(f'  Homogeneity     : {h_ae:.4f}')
print(f'  Completeness    : {c_ae:.4f}')
print()
print('📌 Interpretation:')
print('   → Higher V-Measure = two methods agree on cluster structure')
print('   → If multiple methods agree = clusters are real natural patterns')

In [ ]:
# V-Measure bar chart
vm_data = {
    'vs MeanShift'  : vm_ms,
    'vs DBSCAN'     : vm_db,
    'vs Autoencoder': vm_ae,
}
plt.figure(figsize=(8, 5))
bars = plt.bar(vm_data.keys(), vm_data.values(),
               color=['coral','steelblue','mediumseagreen'], edgecolor='white', width=0.5)
for bar, val in zip(bars, vm_data.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.4f}', ha='center', fontweight='bold', fontsize=11)
plt.ylim(0, 1.1)
plt.ylabel('V-Measure Score')
plt.title('V-Measure Score vs K-Means (reference)\n→ Higher = more agreement between methods', fontweight='bold')
plt.tight_layout()
plt.savefig('data/comparison_vmeasure.png', bbox_inches='tight')
plt.show()

---
## Step 4: Metric Bar Charts — Silhouette & Davies-Bouldin

In [ ]:
algos    = [s['algorithm']      for s in all_scores]
sil_vals = [s['silhouette']     for s in all_scores]
dbs_vals = [s['davies_bouldin'] for s in all_scores]
bar_cols = ['steelblue','coral','mediumseagreen','mediumpurple']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

bars1 = axes[0].bar(algos, sil_vals, color=bar_cols, edgecolor='white', width=0.6)
for bar, val in zip(bars1, sil_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', fontweight='bold', fontsize=10)
axes[0].set_title('Silhouette Score\n(Higher = better separated clusters)', fontweight='bold')
axes[0].set_ylabel('Silhouette Score')
axes[0].set_ylim(0, max(sil_vals) + 0.15)
axes[0].tick_params(axis='x', rotation=15)

bars2 = axes[1].bar(algos, dbs_vals, color=bar_cols, edgecolor='white', width=0.6)
for bar, val in zip(bars2, dbs_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', fontweight='bold', fontsize=10)
axes[1].set_title('Davies-Bouldin Index\n(Lower = better cluster quality)', fontweight='bold')
axes[1].set_ylabel('Davies-Bouldin Index')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('data/comparison_metrics.png', bbox_inches='tight')
plt.show()

---
## Step 5: Side-by-Side 3D Cluster Plots

In [ ]:
all_labels = [km_labels, ms_labels, db_labels, ae_labels]
all_titles = [
    f'K-Means (K={km_scores["n_clusters"]})',
    f'MeanShift ({ms_scores["n_clusters"]} clusters)',
    f'DBSCAN ({db_scores["n_clusters"]} clusters)',
    f'Autoencoder+KMeans (K={ae_scores["n_clusters"]})'
]

fig = plt.figure(figsize=(22, 10))
for idx, (labels, title) in enumerate(zip(all_labels, all_titles)):
    ax = fig.add_subplot(1, 4, idx+1, projection='3d')

    unique = sorted(set(labels))
    n_c    = len(unique) - (1 if -1 in unique else 0)
    pal    = sns.color_palette('tab10', n_c)
    c_map  = {c: pal[i] for i, c in enumerate(l for l in unique if l != -1)}

    # Noise
    if -1 in unique:
        nm = labels == -1
        ax.scatter(df.loc[nm,'Annual Income'], df.loc[nm,'Spending Score'], df.loc[nm,'Age'],
                   c='lightgray', s=3, alpha=0.2, marker='x')

    for c in unique:
        if c == -1: continue
        mask = labels == c
        ax.scatter(df.loc[mask,'Annual Income'], df.loc[mask,'Spending Score'], df.loc[mask,'Age'],
                   color=c_map[c], s=5, alpha=0.5, label=f'C{c}')

    ax.set_xlabel('Income', fontsize=8)
    ax.set_ylabel('Score',  fontsize=8)
    ax.set_zlabel('Age',    fontsize=8)
    ax.set_title(title, fontweight='bold', fontsize=9)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6, loc='upper left')

plt.suptitle('All Algorithms — 3D Cluster Comparison\n→ Compare how each method divides the customer space',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('data/comparison_3d_all.png', bbox_inches='tight')
plt.show()

---
## Step 6: Final Summary & Best Algorithm Decision

In [ ]:
best_sil = algos[sil_vals.index(max(sil_vals))]
best_dbs = algos[dbs_vals.index(min(dbs_vals))]

print('=' * 70)
print('                    FINAL SUMMARY')
print('=' * 70)
print(display_df.to_string(index=False))
print('=' * 70)
print()
print(f'🏆 Best Silhouette Score : {best_sil}')
print(f'🏆 Best Davies-Bouldin   : {best_dbs}')
print()
print('Algorithm Characteristics:')
print('  K-Means          : Needs K upfront. Fast. Good for round clusters.')
print('  MeanShift        : Auto K. Density-based. Can be slow on large data.')
print('  DBSCAN           : Auto K. Finds outliers. Handles any cluster shape.')
print('  Autoencoder+KMeans: Deep learning. Best for complex patterns. Uses epochs.')
print()
overall_best = best_sil  # Change if dbi gives a different winner
print(f'→ SELECTED BEST ALGORITHM: {overall_best}')
print(f'→ This algorithm will be used for the Streamlit UI (app.py)')